<a href="https://colab.research.google.com/github/MUIZ202426/Muiz_Universe/blob/main/Muiz1st_ML_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import cv2
import numpy as np

# 1. Setup Video and Output
video_path = 'traffic.mp4'
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open traffic.mp4. Make sure it is uploaded to Colab!")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Prepare the video writer to save the output
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('project_output.mp4', fourcc, fps, (frame_width, frame_height))

# 2. Use Built-in Background Subtraction (NO XML NEEDED)
bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=40, detectShadows=True)

# 3. Counting Setup
line_height = int(frame_height * 0.65)
offset = 6  # Pixel tolerance for the line
cars_count = 0
matches = []

# Function to find the center of the car
def get_centroid(x, y, w, h):
    cx = int(x + w / 2)
    cy = int(y + h / 2)
    return (cx, cy)

print("Processing video frames using Background Subtraction... Please wait.")

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Skip frames to speed up processing in Colab
    frame_count += 1
    if frame_count % 2 != 0:
        continue

    # --- DETECTION PHASE ---
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Apply the background subtractor to find moving objects
    mask = bg_subtractor.apply(blur)

    # Remove shadows and noise (thresholding and morphology)
    _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Find the borders (contours) of the moving vehicles
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # Draw the main counting line (Yellow)
    cv2.line(frame, (0, line_height), (frame_width, line_height), (0, 255, 255), 2)

    # --- COUNTING PHASE ---
    for contour in contours:
        (x, y, w, h) = cv2.boundingRect(contour)

        # Ignore tiny movements (leaves blowing, small shadows)
        if (w >= 40) and (h >= 40):
            # Draw rectangle around the car
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Get center point and draw it
            centroid = get_centroid(x, y, w, h)
            matches.append(centroid)
            cv2.circle(frame, centroid, 4, (0, 0, 255), -1)

            # Check if the center point crosses our line
            for (cx, cy) in matches:
                if (line_height - offset) < cy < (line_height + offset):
                    cars_count += 1
                    matches.remove((cx, cy))
                    # Flash the line Green when a car is counted
                    cv2.line(frame, (0, line_height), (frame_width, line_height), (0, 255, 0), 3)

    # Display the final count on the screen
    cv2.putText(frame, f"Vehicles Counted: {cars_count}", (30, 60),
                cv2.FONT_HERSHEY_DUPLEX, 1.2, (0, 165, 255), 2)

    # Write frame to output video
    out.write(frame)

# Clean up
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Processing Complete! Total Cars Counted: {cars_count}")
print("Output video saved as 'project_output.mp4'. You can now download it.")


Processing video frames using Background Subtraction... Please wait.
Processing Complete! Total Cars Counted: 91
Output video saved as 'project_output.mp4'. You can now download it.
